# 反哺与结业——Skill 工程与成果评估

> **反哺与结业**（建议 ~1.5h）。产物 ④：Skill diff + evals/门禁验证记录；随后四产物齐检 + 12 项能力地图 + 结业判定。前置要求：03.02/03.03 完成。素材：`src/skill_patch_example.diff`（增补样例，实测依据源自真机记录）。

> **本节新增知识只有 1 件**：**Skill 工程三契约**（frontmatter 定时机 / evals 是门禁 / 静态 UT 保结构）。其余全部复用基础课程知识：Issue→PR 是 02.03 八步与五条红线的原样复用（只是换了个目标仓库）；边界框架回到 02.03 重新检验；产物验收是 01.02 产物走读的进阶。

> **写给第一次接触算子的读者**：算子代码由 AI 团队编写与修复，指挥官角色看得懂结果、做得了决定就够。

## 1. 踩过的坑，还要下一个人再踩一遍吗？

回想 03.02 节：三个坑的排查已完整走过一遍。如果有人把这些经验写成一条策略条目放进 Skill 知识库，此后所有开发者调优时 Agent 都会主动告知。这就是**反哺生态**：把个人经验回馈给所有用户，是从学员到贡献者的最后一步。

这套反哺在生态里的正式图景是**双流工作流**，三句话说清：**训练流**生产知识（全流程验证后方可录入，杜绝未验证的经验入库）；**推理流**执行调优（五阶段闭环，策略库是它的输入）；实现不出的**模板缺口回流训练流**补充，10.3 要做的增补策略条目正是这个回流动作。
<img src="./images/03.04_dual_flow.png" alt="双流工作流闭环：训练流生产知识 ↔ 推理流执行调优，模板缺口回流" width="700px">


## 2. Skill 工程解剖

> **术语卡**：
>
> | 术语 | 一句话 | 类比 |
> |---|---|---|
> | **SKILL.md** | Skill 主文件：说明 + 路由 | 手册目录页 |
> | **frontmatter** | 文件头元信息，决定被调用时机 | 索引标签 |
> | **evals** | 评测契约（每款 Skill 必有） | 质检标准 |

```text
cannbot-skills/ops/ascendc-performance-best-practices/
├── SKILL.md            # 算子族总表（路由层）
├── references/
│   ├── elementwise/    # 逐元素族（今天修改对象）
│   ├── broadcast/ ...  # 其他族
└── evals/evals.json    # 8 case
```

**三个关键契约**（Skill 的"需求五要素 + 验收门禁"，结构与基础课程 01.02 的算子工程同构）：

| 契约 | 作用 | 违约后果 |
|---|---|---|
| frontmatter（文件头说明：我是谁、何时用）的 description/trigger | 决定 Agent 是否在**正确时机**加载 | 时机错：需要时没加载，不需要时被塞上下文 |
| evals/evals.json（评测用例集：给 Skill 出的考试卷） | 仓级硬门禁（S-EVAL-01） | 无法回归验证 Skill 行为，不予合入 |
| `bash tests/run-tests.sh --fast` | 全仓静态结构检查（CI 硬门禁） | 断链/字段缺失直接挡 PR |

**知识条目的可用性分三级**（实施专家视角）：✅ 可直接拷贝（完整 kernel（算子核心代码）模板，如 `broadcast_common.h`）· ⚠️ 部分实现（仅设计参考，按范式改写）· ❌ 未收录（自行实现并**标记缺口，反哺训练流**，这是双流闭环的入口）。

一条心法：**知识 > 流程复杂度**：增加知识库的收益远胜增加工作流复杂度；补一块知识，下一轮同类算子全受益。

三个关键契约看完，运行下面的 cell，亲手校验一份 evals 契约的完整性（这正是门禁 S-EVAL-01 检查的内容）：


In [ ]:
# evals 契约校验：加载内置参考，检查 schema 完整性
import json, os

paths = ['src/evals_reference.json']
data = None
for p in paths:
    if os.path.exists(p):
        data = json.load(open(p)); break

if data is None:
    print("（内置 evals_reference.json 未找到，展示契约结构）")
    print("═" * 50)
    print('  {"eval_mode": "text", "skill_name": "...",')
    print('   "evals": [{"id": 1, "title": "...", "config": "...",')
    print('              "prompt": "...", "expected_output": "...",')
    print('              "expectations": [...], "files": [...]}]}')
else:
    print(f"evals 契约校验（{len(data['evals'])} cases）")
    print("═" * 50)
    required = ['id', 'title', 'prompt', 'expected_output']
    all_ok = True
    for e in data['evals']:
        missing = [k for k in required if k not in e or not e[k]]
        status = "✓" if not missing else f"✗ 缺 {missing}"
        if missing: all_ok = False
        print(f"  [{status}] case {e.get('id', '?')}: {e.get('title', '?')[:35]}")
    print("═" * 50)
    print(f"  {'✓ 契约完整' if all_ok else '✗ 有缺字段'}")

## 3. Lab D：增补策略条目（25min）

### 准备：获取 cannbot-skills 工作副本

```bash
git clone https://gitcode.com/cann/cannbot-skills.git
cd cannbot-skills && git checkout 94216ba2
```

### 任务 A（15min）：增补精度预算策略条目

对照 `src/skill_patch_example.diff`（把 gelu 的 8 步链精度预算经验写入 elementwise 族）。然后写下你自己的一条：从 03.02/03.03 挑一条经验（候选：Compare 256B 注意事项、计算密度分析方法、launch 地板判据），按实测依据 → 策略 → 纪律的三段式增补。

### 任务 B（10min）：验证未破坏

```bash
python3 -c "import json; d=json.load(open('ops/ascendc-performance-best-practices/evals/evals.json')); print(len(d['evals']), 'cases OK')"
bash tests/run-tests.sh --fast
```

## 4. 贡献路径（口头，5min）

Skill diff → evals 通过 → run-tests 通过 → 九维标准自检（cannbot-skill-reviewer）→ Issue → PR → 合入。这是 01.03 §3.5 贡献入口的实操版。

## 5. 要点回顾

**本章数字锚点**：1 个双流闭环 · 3 条契约 · 1 套分级

- Skill 三契约：**frontmatter 定时机 · evals 是门禁 · 静态 UT 保结构**；**双流闭环**（新）：训练流沉淀 ↔ 推理流消费，缺口回流；
- 知识条目分级（新）：✅ 拷贝 · ⚠️ 参考 · ❌ 缺口即反哺入口；
- 策略条目三段式：**实测依据 → 规则 → 纪律**；心法：**知识 > 流程复杂度**。

## 6. 带走物

| 类别 | 内容 |
|---|---|
| **一句话** | 个人经验就是生态的知识，一条策略条目让所有人受益 |
| **工具** | Skill 三契约 + cannbot-skill-reviewer 九维评审 + 双流闭环 |
| **产物** | Skill diff + evals/门禁验证记录 + 四件产物全验收 |
| **模板** | 策略条目三段式（实测依据→策略→纪律）；Issue→PR 八步与五条红线 |
| **下一步** | cann-bench 评测 / Skill PR 提交 / [专题课件](../README.md)按需选学 |

## 7. 章末检查点

1. **（单选）** Skill 的 description/trigger 最直接决定什么？
   - A. 文档渲染效果  B. Agent 是否在正确时机加载  C. evals 通过率  D. CI 速度

2. **（简答）** 把今天的一条经验写成策略条目时，实测依据为什么必须写？

<details><summary>参考答案</summary>

1. **B** 2. 可溯源性：策略条目是给未来用户的断言，标注实测边界才能判断适用范围。
</details>

## 8. 产物自查表（交付 ④）

| 项 | 标准 | 自查 |
|---|---|---|
| 条目质量 | 实测依据 + 规则 + 纪律三段式 | ☐ |
| 门禁验证 | evals 合法 + run-tests 全绿 | ☐ |
| 可评审性 | 口头复述九维/回流路径 | ☐ |

---

产物 ④ 落定，四件产物至此齐备。最后一程：终检、能力地图与结业。


---

> **成果评估与结业**（约 30min）。

## 9. 四件产物齐检（10min）

逐项检查四件产物。这正是基础课程 01.02 产物走读的终局版：那时读别人的，现在验收自己一天的交付：

| # | 产物 | 来自 | 关键验收点 |
|---|---|---|---|
| ① | gelu_approx 工程 + 测试证据 | 03.01 | 四件套可构建 · 全量绿 · 台账完整 |
| ② | 三坑修复三件套 | 03.02 | 根因 + diff + 回归 · 证据链可回溯 |
| ③ | 性能对比报告 | 03.03 | 逐用例 · 计算密度分析 · 双回归 |
| ④ | Skill diff + 验证记录 | 03.04 | 门禁绿 · 可评审 |


**本章数字锚点**：4 件产物 · 12 项能力

## 10. 12 项能力地图自评（10min）

### 来自基础课程的能力（6 项，括注册号出处）

| # | 能力 | 我能做到吗 |
|---|---|---|
| 1 | 用 Flash 版独立交付简单算子（01.02） | ☐ |
| 2 | 说清 Plugin/Agent/Skill 三层架构（01.01） | ☐ |
| 3 | 按症状路由到正确的调试 Skill（01.03） | ☐ |
| 4 | 解释 v3.0.0 三大设计思想（02.01） | ☐ |
| 5 | 说出性能三原则并解释为什么平均值会掩盖退化（02.02） | ☐ |
| 6 | 知道 Issue→PR 的基本流程与红线（02.03） | ☐ |

### 下半场新增的高级能力（6 项，对应 07~10）

| # | 能力 | 我能做到吗 |
|---|---|---|
| 7 | **多步计算链的精度预算**：估算 N 步链的舍入累积并制定分段策略 | ☐ |
| 8 | **链路级调试**：分段 dump 定位是哪一步引入误差 | ☐ |
| 9 | **计算密度分析**：判断访存受限假设是否成立 | ☐ |
| 10 | **Skill 工程规范**：能增补条目并过 evals/门禁 | ☐ |
| 11 | **判读 v3.0.0 验收报告**：从证据到结论的完整链路 | ☐ |
| 12 | **理解生态闭环**：一线经验如何变成全社区复用的知识 | ☐ |

答完 12 项后，运行下面的判分 cell 统计你的掌握情况：


In [ ]:
# 12 项能力地图自评：填 True/False，运行得分
my_abilities = {
    # 阶段一/二基础能力（6 项）
    1: True,   # Flash 版独立交付
    2: True,   # 三层架构
    3: True,   # 调试路由
    4: True,   # v3.0.0 三大思想
    5: True,   # 性能三原则
    6: True,   # Issue→PR 流程
    # 今天新增能力（6 项）
    7: True,   # 精度预算
    8: True,   # 链路级调试
    9: True,   # 计算密度
    10: True,  # Skill 工程
    11: True,  # 验收报告判读
    12: True,  # 生态闭环理解
}
base = sum(1 for k in range(1, 7) if my_abilities[k])
adv = sum(1 for k in range(7, 13) if my_abilities[k])
total = base + adv
print("能力地图自评")
print("═" * 40)
print(f"  基础能力: {base}/6")
print(f"  高级能力: {adv}/6")
print(f"  总分: {total}/12")
print("═" * 40)
weak = [k for k, v in my_abilities.items() if not v]
if weak:
    print(f"  薄弱项: {weak}")
    print("  → 回看对应模块补强")
else:
    print("  ✓ 全部通过！")

## 11. 结业判定、AI 边界与后续路径（10min）

**结业标准**：四件产物齐检全部通过（产物 ④ 为必要条件）。

**结业前的最后一课：回到基础课程 02.03 的边界框架**。基础课程（02.03 §3）曾从听过的认知接受了那张 ✓ 适合交给 Agent / ✗ 人主导的框架，现在带着**四件产物**回来逐条重新检验：

| 框架条目（基础课程 02.03） | 本路径的亲历证据 |
|---|---|
| ✓ 验收标准明确（有 golden（参考答案）可对拍） | gelu 有 golden 有容差 → 全自动交付成立 |
| ✓ 流程成熟（PR/Issue 协作） | 本节 3 的 Issue→PR 沿用 02.03 八步与红线 |
| ✓ 重复性高（同类算子开发） | elementwise 家族策略直接复用（双流闭环） |
| ✗ 人主导：合入决策、探索性架构 | 你做的分段阈值确认、03.03 节的"何时不该优化"判定 |

**gelu_approx 能全自动，正因为它四条全占**，而阈值决策与停止判断留给了开发者。

**团队引入看 ROI，三步走**：小场景试点（高频 + 有明确验收）→ 双重验证（精度 + 质量门禁）评估→ 有效再全流程推广。

**后续路径**：
- **cann-bench 实测**：把你开发的 gelu_approx 提交到公共评测平台
- **社区贡献**：把 03.04 节的 Skill diff 整理为 Issue → PR 提交到 cannbot-skills
- **轨道课件深入**：[阶段二轨道](../README.md)（路径精通 / 工程质量 / 性能深潜 / 950 新硬件）
- **完整体系回顾**：[CANNBot 系列课程总览](../README.md)

不确定后续方向？运行下面的路径推荐器：


In [ ]:
# 后续路径推荐器（仿 01.03 learning_path）
def next_step(interest):
    """根据兴趣推荐后续学习/贡献路径"""
    paths = {
        "性能": ["cann-bench 提交 gelu_approx 评测", "阶段二轨道 C（排序优化）", "ops-profiling 深入"],
        "贡献": ["Skill diff → Issue → PR 提交到 cannbot-skills", "九维标准自检", "加入 SIG 讨论"],
        "新硬件": ["阶段二轨道 D（RegBase + SIMT）", "dav-3510/950 路径尝试"],
        "模型": ["model-infer-* 系列教程", "llm_inference 课程"],
    }
    return paths.get(interest, ["CANNBot 系列课程总览", "按兴趣选择方向"])

print("后续路径推荐")
print("═" * 40)
for interest in ["性能", "贡献", "新硬件"]:
    recs = next_step(interest)
    print(f"\n  ▸ 如果你对「{interest}」感兴趣：")
    for i, r in enumerate(recs, 1):
        print(f"    {i}. {r}")

> **结业讨论（自由进行，不计时）**。So what：三个阶段里，最值得立刻拿走的一个机制是什么？Now what：回去这周，打算用 CANNBot 做的第一件**具体**的事是什么（具体到算子名或场景）？把它写在产物自查表背面，这就是行动承诺。

## 12. 结课寄语

今天走完了生成、调试、优化、反哺的完整循环，CANNBot 生态就靠这个循环运转：**一线实测经验 → Skill 知识 → 全体用户复用**。

你的四件产物，就是你在 CANNBot 生态的第一批贡献。

## 13. 课后练习

> 练习答案：[`answer/03.04_answer.txt`](./answer/03.04_answer.txt)

1. **（单选题）** 双流工作流中，模板缺口回流训练流发生在什么时机？
   - A. 每次性能验证通过后自动触发
   - B. 推理流的实施专家实现不出模板、知识库也无收录时，标记缺口，反哺训练流补充
   - C. 训练流开始前的初始化阶段
   - D. evals 门禁失败时

2. **（简答）** 用本节 3 增补的策略条目为例，说明知识 > 流程复杂度：这条知识会被谁、在推理流的哪一步用到？如果它不存在，流程要多复杂才能弥补？

3. **（编程实践·必做）** **从学员到贡献者**：把你当天的一条实测经验（候选：Compare 256B 注意事项 / 跷跷板效应与验收标准 / 计算密度分析方法）按实测依据 → 策略 → 纪律的三段式写成策略条目，跑 evals + `run-tests.sh --fast` 验证未破坏。**若质量达标，按 02.03 的八步与五条红线整理为 Issue → PR 提交到 cannbot-skills**。内容源于代码、写后 GET 回查，全程适用。

4. **（编程实践·选做）** **AI 边界评估**：选你团队的一个真实场景，用本节 10 的决策框架给出适合、不适合交给 Agent 各一条判据（对照四条属性），并写出 ROI 三步走的试点计划（试点场景 → 双重验证标准 → 推广条件）。
